# Website Summarizer with OpenAI

This notebook fetches text from a webpage, cleans it, and generates a summary using the OpenAI chat Completions API.

## Goals 
- Learn how to retrieve website content
- Clean HTML into plain text
- Send content to an LLM for summarization
- Understand the limits of direct summrization


In [ ]:
%pip install openai python-dotenv requests beautifulsoup4

In [2]:
import os
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [3]:
## Load environment variables
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("No OpenAI API key found in environment variables")

client = OpenAI(api_key=api_key)
print("OpenAI client initialized successfully")

OpenAI client initialized successfully


In [19]:
## Fetch the webpage content
import requests
## Define the URL to scrape
url = "https://en.wikipedia.org/wiki/Nepal"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers, timeout=10)
response.raise_for_status()

html = response.text
print(f"Fetched {len(html)} characters")

Fetched 1160341 characters


In [20]:
from bs4 import BeautifulSoup
import re

soup = BeautifulSoup(html, "html.parser")

# 1. Remove unwanted tags
for tag in soup(["script", "style", "noscript", "template", "iframe", "sup"]):
    tag.decompose()

# 2. Target main content (Wikipedia specific)
main_content = soup.find("div", {"id": "mw-content-text"})

if not main_content:
    raise ValueError("Main content not found")

# 3. Extract only meaningful text blocks
paragraphs = main_content.find_all(["p", "h2", "h3"])

clean_text = []

for p in paragraphs:
    text = p.get_text(" ", strip=True)
    
    # Remove citation markers like [1], [23]
    text = re.sub(r"\[\d+\]", "", text)
    
    # Skip empty or very short lines
    if len(text) > 50:
        clean_text.append(text)

# 4. Join into final text
final_text = "\n\n".join(clean_text)

print(final_text[:5000])  # limit output

Nepal , officially the Federal Democratic Republic of Nepal , is a landlocked country in South Asia . It is mainly situated in the Himalayas , but also includes parts of the Indo-Gangetic Plain . It borders the Tibet Autonomous Region of China to the north , and India to the south, east, and west , while it is narrowly separated from Bangladesh by the Siliguri Corridor , and from Bhutan by the Indian state of Sikkim . Nepal has a diverse geography , including fertile plains , subalpine forested hills, and eight of the world's ten highest mountains , including Mount Everest , the highest point above mean sea level on Earth. Kathmandu is the nation's capital and its largest city . Nepal is a multi-ethnic, multi-lingual, multi-religious, and multi-cultural sovereign state, with Nepali as the official language.

The name "Nepal" is first recorded in texts from the Vedic period of the Indian subcontinent , the era in ancient Nepal when Hinduism was founded, the predominant religion of the c

In [21]:
## Create a summary prompt
prompt = f"""
Summarize the following webpage content clearly and concisely.

Focus on:
- the main topic
- key points
- any important conclusions or takeaways

Webpage content:
{final_text}
"""



In [ ]:
## Call openAI
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that summarizes text."},
        {"role": "user", "content": prompt}
    ],
    temperature=0.3,
)
summary = response.choices[0].message.content
print(summary)

**Main Topic: Overview of Nepal**

**Key Points:**

1. **Geography and Demographics:**
   - Nepal is a landlocked country in South Asia, primarily located in the Himalayas, featuring diverse geography including fertile plains and eight of the world's ten highest mountains, including Mount Everest.
   - The capital is Kathmandu, and the country is multi-ethnic, multi-lingual, and multi-religious, with Nepali as the official language.

2. **Historical Context:**
   - The name "Nepal" has ancient roots, with references in Vedic texts and connections to Hinduism and Buddhism.
   - Nepal was unified in the 18th century under the Gorkha Kingdom and has a history of monarchy, colonial resistance, and civil conflict, leading to the establishment of a federal democratic republic in 2008.

3. **Political Structure:**
   - Nepal is a federal parliamentary republic divided into seven provinces, with a multi-party system. The Constitution of Nepal was adopted in 2015, affirming its secular and demo

In [25]:
## Display the summary
display(Markdown(" ## Summary"))
display(Markdown(summary))

 ## Summary

**Main Topic: Overview of Nepal**

**Key Points:**

1. **Geography and Demographics:**
   - Nepal is a landlocked country in South Asia, primarily located in the Himalayas, featuring diverse geography including fertile plains and eight of the world's ten highest mountains, including Mount Everest.
   - The capital is Kathmandu, and the country is multi-ethnic, multi-lingual, and multi-religious, with Nepali as the official language.

2. **Historical Context:**
   - The name "Nepal" has ancient roots, with references in Vedic texts and connections to Hinduism and Buddhism.
   - Nepal was unified in the 18th century under the Gorkha Kingdom and has a history of monarchy, colonial resistance, and civil conflict, leading to the establishment of a federal democratic republic in 2008.

3. **Political Structure:**
   - Nepal is a federal parliamentary republic divided into seven provinces, with a multi-party system. The Constitution of Nepal was adopted in 2015, affirming its secular and democratic nature.

4. **Foreign Relations:**
   - Nepal maintains a policy of neutrality and has established diplomatic relations with numerous countries, emphasizing cooperation with India and China, and is a founding member of SAARC.

5. **Economy:**
   - Nepal is one of the least developed countries, heavily reliant on agriculture, remittances, and tourism, with significant contributions from foreign aid.
   - The economy faces challenges such as poor infrastructure, political instability, and vulnerability to natural disasters.

6. **Biodiversity and Environment:**
   - The country is rich in biodiversity, hosting a variety of ecosystems and endangered species, but faces environmental challenges due to human encroachment and climate change.

7. **Culture and Society:**
   - Nepal has a rich cultural heritage with diverse traditions, languages, and religions. Major festivals reflect its Hindu and Buddhist influences.
   - Traditional clothing, cuisine, and music vary significantly across different ethnic groups, showcasing the country's cultural diversity.

8. **Education and Health:**
   - Literacy rates have improved, but challenges remain in access to education and healthcare, particularly in rural areas.
   - Public health initiatives have made progress in maternal and child health, though malnutrition and communicable diseases persist.

9. **Sports and Recreation:**
   - Football and cricket are popular sports, with volleyball declared the national sport. Traditional games and sports are also prevalent, although infrastructure and funding for sports development are limited.

**Conclusions/Takeaways:**
- Nepal is a country of rich cultural diversity and natural beauty, facing significant socio-economic challenges.
- The transition to a federal democratic republic marks a pivotal moment in its history, with ongoing efforts to improve governance, economic stability, and social equity.
- The country's unique geography and biodiversity present both opportunities and challenges for sustainable development.

In [26]:
## Show token usage
if response.usage:
    print(f"Tokens used: {response.usage.total_tokens}")
    print(f"Prompt tokens: {response.usage.prompt_tokens}")
    print(f"Completion tokens: {response.usage.completion_tokens}")
    print(f"Total cost: ${response.usage.total_tokens / 1000000 * 0.15:.6f}")
    


Tokens used: 18310
Prompt tokens: 17728
Completion tokens: 582
Total cost: $0.002746
